# Phase 3 - Skin-Condition Classifier Training
## Google SCIN + Fitzpatrick17k via HuggingFace (streaming)

**Key fix:** `streaming=True` means NO 13 GB download.
Images are processed one at a time directly from HuggingFace parquet shards.
Free Colab (15 GB disk) works fine.

**One-time setup before running:**
1. Accept SCIN terms at https://huggingface.co/datasets/google/scin
2. Accept Fitzpatrick17k (search `mattgroh/fitzpatrick17k` on HF)
3. Get a **read token** from https://huggingface.co/settings/tokens
4. Push repo to GitHub, fill `GITHUB_USER` in Cell 1

| Source | Total images | After face/neck filter |
|---|---|---|
| `google/scin` | ~10k | ~1-3k |
| `fitzpatrick17k` | ~16k | ~2-4k |
| **Combined** | **~26k scanned, ~13 GB streamed** | **~3-7k saved** |

## 0. Confirm GPU

In [ ]:
!nvidia-smi | head -10
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 1. Clone or update repo + install

In [ ]:
GITHUB_USER = "moodykhalif23"
REPO_NAME   = "facemeup"            # actual GitHub repo name
BRANCH      = "main"
REPO_URL    = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
LOCAL_DIR   = "/content/skincare"

import os, subprocess

if os.path.isdir(LOCAL_DIR + '/.git'):
    # Repo already cloned — pull latest changes
    print("Repo exists — pulling latest changes...")
    r = subprocess.run(['git', '-C', LOCAL_DIR, 'pull', '--ff-only'],
                       capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip())
    # Clear stale .pyc files so Python picks up updated source
    subprocess.run(['find', LOCAL_DIR, '-name', '*.pyc', '-delete'], capture_output=True)
    subprocess.run(['find', LOCAL_DIR, '-name', '__pycache__', '-type', 'd',
                    '-exec', 'rm', '-rf', '{}', '+'], capture_output=True)
    print('cache cleared')
else:
    print(f'Cloning {REPO_URL}...')
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, LOCAL_DIR], check=True)

%cd /content/skincare/backend_v2
import sys, os

!pip install -q -e ml_service
!pip install -q -e ml_training

# Pip editable installs in Colab subprocesses are unreliable — add both
# source roots to sys.path and PYTHONPATH explicitly.
# ml_service:   importable as 'app'         (source root = ml_service/)
# ml_training:  importable as 'skin_training' (source root = ml_training/src/)
ML_SERVICE  = os.path.abspath('ml_service')
ML_TRAIN_SRC = os.path.abspath('ml_training/src')
for p in [ML_SERVICE, ML_TRAIN_SRC]:
    if p not in sys.path: sys.path.insert(0, p)
PYTHONPATH = f'{ML_SERVICE}:{ML_TRAIN_SRC}'
os.environ['PYTHONPATH'] = PYTHONPATH
print('PYTHONPATH:', PYTHONPATH)

ok = True
for pkg in ['app.config', 'skin_training.data.sources.scin_hf']:
    try:
        __import__(pkg)
        print(f'OK  {pkg}')
    except ImportError as e:
        print(f'FAIL {pkg}: {e}')
        ok = False
if not ok:
    raise RuntimeError('Import check failed — see FAIL lines above')

## 2. Set HuggingFace token

> **Why not `login()`?**  
> When running Colab from the VS Code extension, the vault secret injection
> is unavailable. Set the token directly via `os.environ` instead.

Paste your **read token** from https://huggingface.co/settings/tokens
(it starts with `hf_`)

In [ ]:
import os
HF_TOKEN = 'hf_PASTE_YOUR_TOKEN_HERE'   # <-- replace this

os.environ['HF_TOKEN'] = HF_TOKEN

# Verify it works
from huggingface_hub import whoami
try:
    info = whoami(token=HF_TOKEN)
    print('Logged in as:', info['name'])
except Exception as e:
    print('Auth error:', e)
    print('Check your token at https://huggingface.co/settings/tokens')

## 3. Inspect SCIN schema + face-region coverage

SCIN uses **multi-hot boolean columns** for body parts:
  `body_parts_head_or_neck`, `body_parts_arm`, `body_parts_palm`, etc.
This cell shows which columns exist and how many rows are face/neck.

In [ ]:
from datasets import load_dataset

ds = load_dataset("google/scin", split="train", streaming=True, token=HF_TOKEN)

cols = set(ds.features.keys())
print('--- All columns ---')
for col in sorted(cols):
    print(f"  {col}")

# SCIN multi-hot body-part columns
body_cols = sorted(c for c in cols if c.startswith('body_parts_'))
FACE_BP   = {'body_parts_head_or_neck','body_parts_face','body_parts_scalp','body_parts_neck'}
face_cols = [c for c in body_cols if c in FACE_BP]
print(f'\nFace body-part columns found: {face_cols}')
print(f'All body-part columns: {body_cols}')

# Sample first 500 rows to show face-region rate
face_n = total_n = 0
for i, row in enumerate(ds):
    if i >= 500: break
    total_n += 1
    if face_cols and any(str(row.get(c,'')).lower() in ('true','1') for c in face_cols):
        face_n += 1
    elif not face_cols:
        face_n += 1   # no body-part info — count all
print(f"\nFirst 500 rows: {face_n} face/neck ({100*face_n/max(1,total_n):.0f}%)")
print("Extrapolating to full 10k dataset: ~", round(face_n/500*10000), "face images")

## 4. Precompute aligned face tensors  (~20-60 min)

Streams from HuggingFace, filters face/neck/scalp rows, runs:
`decode -> face-detect -> align -> CLAHE+WB -> save .npy`

Disk usage: only the .npy files (~200 MB for 1k faces, ~600 MB for 3k).
No parquet shards saved locally.

| `--source` | What streams |
|---|---|
| `scin_hf` | SCIN only |
| `fitzpatrick17k` | Fitzpatrick17k only |
| `all` | Both combined (recommended) |

Fully resumable - re-run to continue.

In [ ]:
import os
SOURCE = 'all'   # recommended: combine SCIN + Fitzpatrick17k for a usable train split

# Both paths needed: ml_service ('app') and ml_training/src ('skin_training')
PYPATH = f"{os.path.abspath('ml_service')}:{os.path.abspath('ml_training/src')}"
os.environ['PYTHONPATH'] = PYPATH

!PYTHONPATH={PYPATH} python -m skin_training.data.precompute \
  --source {SOURCE} \
  --output /content/work \
  --aligned-size 256 \
  --verbose 2>&1 | tail -80

In [ ]:
import json, re, pandas as pd
from pathlib import Path
index_path = Path('/content/work/index.json')
labels_path = Path('/content/work/labels.csv')
if not index_path.is_file() or not labels_path.is_file():
    raise FileNotFoundError(
        'Phase 4 did not produce /content/work/index.json and labels.csv. '
        'Scroll up to the precompute cell, fix that error, then rerun phase 4 before training.'
    )
idx = json.loads(index_path.read_text())
print(json.dumps(idx, indent=2))
n, pct = idx['n_samples'], idx.get('face_detect_skip_pct','?')
print(f"\nAligned face tensors: {n}  |  face-detector skip rate: {pct}%")
df = pd.read_csv(labels_path)
cond_cols = [c for c in df.columns if re.match(r'^c\d+_', c)]
print("\nCondition coverage:")
for col in cond_cols:
    n_pos = int(df[col].sum())
    pct2  = 100 * n_pos / max(1, len(df))
    flag  = "[!] " if n_pos < 30 else "    "
    print(f"  {flag}{col:<28} {n_pos:>6} positives  ({pct2:.1f}%)")
if n < 300:
    print("\n[!] Very few samples - try --all-body-parts flag")

## 5. Configure training

In [ ]:
import os, yaml
from pathlib import Path
cfg = yaml.safe_load(Path('ml_training/configs/base.yaml').read_text())
n_samples = int(idx['n_samples'])
estimated_train = max(1, int(round(n_samples * (1 - cfg['data']['val_split'] - cfg['data']['test_split']))))
cfg['data']['aligned_dir']          = '/content/work'
cfg['data']['num_workers']          = min(2, os.cpu_count() or 2)
cfg['train']['checkpoint_dir']      = '/content/work/runs/exp1'
cfg['train']['batch_size']          = 32
if estimated_train < cfg['train']['batch_size']:
    cfg['train']['batch_size'] = max(2, min(8, estimated_train))
cfg['train']['epochs']              = 25
cfg['train']['mixed_precision']     = True
cfg['train']['early_stop_patience'] = 6
print(f"n_samples={n_samples} estimated_train={estimated_train} batch_size={cfg['train']['batch_size']} num_workers={cfg['data']['num_workers']}")
Path('/content/config.yaml').write_text(yaml.safe_dump(cfg))
print(yaml.safe_dump(cfg))

## 6. Train  (~40-80 min on T4)
Resumable - interrupt and re-run to continue from last.pt.

In [ ]:
!python -m skin_training.train.loop --config /content/config.yaml --verbose

## 7. TensorBoard (run in a second tab while training)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/work/runs/exp1/tb

## 8. Export ONNX + roundtrip check

In [ ]:
from pathlib import Path
checkpoint = Path('/content/work/runs/exp1/best.pt')
if not checkpoint.is_file():
    fallback = Path('/content/work/runs/exp1/last.pt')
    if fallback.is_file():
        print(f'best.pt missing, falling back to {fallback}')
        checkpoint = fallback
    else:
        raise FileNotFoundError('Training produced neither best.pt nor last.pt')
!python -m skin_training.export.to_onnx \
  --checkpoint {checkpoint} \
  --config /content/config.yaml \
  --output /content/work/skin_classifier_mobilenet.onnx \
  --verbose

## 9. Save to Drive + download
Drive is only used here (artifact output), never for input data.

In [ ]:
import shutil
from pathlib import Path
from google.colab import drive, files
artifacts = [
    Path('/content/work/skin_classifier_mobilenet.onnx'),
    Path('/content/work/runs/exp1/best.pt'),
    Path('/content/work/index.json'),
]
try:
    drive.mount('/content/drive')
    out_dir = Path('/content/drive/MyDrive/skincare/artifacts')
    out_dir.mkdir(parents=True, exist_ok=True)
    for artifact in artifacts:
        if artifact.is_file():
            shutil.copy2(artifact, out_dir / artifact.name)
    print('saved available artifacts to Drive')
except Exception as e:
    print('Drive mount skipped or failed:', e)
for artifact in artifacts:
    if artifact.is_file():
        print('artifact:', artifact)
if artifacts[0].is_file():
    files.download(str(artifacts[0]))

## 10. Local install  (Windows machine)

```powershell
Copy-Item $HOME\Downloads\skin_classifier_mobilenet.onnx `
  C:\Users\Sozuri\skincare\backend_v2\ml_service\models\

cd C:\Users\Sozuri\skincare\backend_v2
docker compose build ml-service
docker compose up -d ml-service

curl http://localhost:8013/healthz   # expect models_loaded: true
```

After that POST /api/v1/analyze returns `"inference_mode": "onnx_mobilenet"`.

Report back: val_F1_macro + Fitzpatrick stratified block + index.json
Then I start Phase 6: MobileNet distillation + INT8 quant.